# Floating Point Is Logarithmic: `log1p`, `logsumexp` and Numerical Stability

A computer's numbers are a logarithmic ruler: they are stored as a mantissa times a power of two, so they are spaced by **ratios**, not by steps. This explains both why floats can span $10^{-308}$ to $10^{308}$ and why products of many small probabilities collapse to zero. The cure, again, is a logarithm — used carefully, with the special functions `log1p`, `expm1` and `logsumexp`.

This is notebook 04 of module 03 (**Computing logarithms**) in the **Learning Logarithms** series.

## 1. Introduction

A spam filter, a speech recogniser and a DNA aligner all compute the probability of a long sequence of independent events — a product of many numbers between $0$ and $1$. Try it with $1000$ rolls of a fair die:

$$
P = \left(\tfrac16\right)^{1000} \approx 10^{-778} .
$$

```python
>>> (1/6) ** 1000
0.0
```

The true value is not zero, but the smallest positive float is about $5 \times 10^{-324}$, so the product **underflows**. Every comparison between two such models ("is this sequence more likely under model A or model B?") becomes $0$ versus $0$. Sums of exponentials overflow in the other direction: $e^{1000}$ is `inf`.

The fix is the logarithm's oldest trick: work with $\ln P = \sum \ln p_i$, a perfectly ordinary number ($1000 \ln \tfrac16 \approx -1791.76$). But logs bring their own hazards — $\ln(1 + x)$ for tiny $x$ loses all its digits, and adding probabilities in the log domain needs care. This notebook explains why, using the structure of floating-point numbers, and shows the standard tools.

**Prerequisites**

- How a float is split into $m \cdot 2^k$, and ulps — [How computers compute logs](03_how_computers_compute_logs.ipynb).
- The series $\ln(1 + x) = x - x^2/2 + \cdots$ — [Series for the logarithm](../01_the_natural_logarithm/04_series_for_the_logarithm.ipynb).
- Products of probabilities for independent events — [Independence and the product rule](../../../LearningProbability/coursera/00_introduction_to_probability/06_independence_product_rule.ipynb) (LearningProbability).
- Softmax, where `logsumexp` is used in practice — [Softmax and cross-entropy](../../../LearningNeuralNetworks/prerequisites/07_softmax_and_cross_entropy.ipynb) (LearningNeuralNetworks).

After completing this notebook you will be able to:

- Describe a double-precision float as $\pm(1.f)_2 \cdot 2^{e}$ and explain why floats are evenly spaced on a **logarithmic** axis.
- Use `np.spacing`, `np.nextafter` and `np.finfo` to find machine epsilon, the smallest normal and subnormal numbers, and the largest float.
- Explain catastrophic cancellation in $\ln(1 + x)$ and $e^{x} - 1$, and fix it with `math.log1p` and `math.expm1`.
- Implement `logsumexp` with the max-shift trick and compare it with the naive formula and `np.logaddexp`.
- Multiply thousands of probabilities safely in the log domain.

## 2. Theory

### 2.1 A float is a mantissa times a power of two

A 64-bit IEEE 754 double stores a sign bit, an 11-bit exponent and a 52-bit fraction $f$, and represents

$$
x = \pm\, (1.f)_2 \cdot 2^{e}, \qquad 1 \le (1.f)_2 < 2, \qquad -1022 \le e \le 1023 .
$$

Taking $\log_2$ — exponent form and log form side by side:

| Exponent form | Log form |
|---|---|
| $x = (1.f)_2 \cdot 2^{e}$ | $\log_2 x = e + \log_2 (1.f)_2$, with $0 \le \log_2(1.f)_2 < 1$ |
| $x \cdot 2 = (1.f)_2 \cdot 2^{e+1}$ | $\log_2(2x) = \log_2 x + 1$ |

The exponent $e$ is (almost) the integer part of $\log_2 x$. A float is a number written in scientific notation with base $2$ — the slide-rule user's habit of tracking digits and power of ten separately (notebook 02), done in hardware.

### 2.2 Floats are evenly spaced in ratio

Within one **binade** $[2^e, 2^{e+1})$ there are exactly $2^{52}$ equally spaced floats, with spacing

$$
\operatorname{ulp}(x) = 2^{e - 52} .
$$

The next binade has the same number of floats spread over twice the length, so the spacing doubles. The **relative** spacing $\operatorname{ulp}(x)/x$ therefore stays between $2^{-53}$ and $2^{-52}$ everywhere: floats have a fixed number of significant digits (about $15.95$), not a fixed number of decimal places. On a **logarithmic** axis they are almost evenly spaced — the same constant relative precision as the slide rule's scales.

Important landmarks:

- **machine epsilon** $\varepsilon = 2^{-52} \approx 2.22 \times 10^{-16}$: the gap between $1$ and the next float (`np.nextafter(1.0, 2.0) - 1`);
- largest float $\approx 1.80 \times 10^{308}$; anything bigger **overflows** to `inf`;
- smallest **normal** float $2^{-1022} \approx 2.23 \times 10^{-308}$;
- below that, **subnormal** floats (leading bit $0$, fewer significant digits) go down to $2^{-1074} \approx 4.9 \times 10^{-324}$; anything smaller **underflows** to $0$.

In log form the representable range is just $\log_{10} x \in [-323.3, 308.3]$. A product of probabilities whose $\log_{10}$ is $-778$ has nowhere to go — but the number $-778$ itself is perfectly representable. That is the whole case for working in logs.

### 2.3 Catastrophic cancellation: $\ln(1 + x)$ and $e^x - 1$

For tiny $x$, $\ln(1 + x) \approx x$. But a computer first forms $1 + x$, which is rounded to the nearest float near $1$ — a grid with spacing $2.2 \times 10^{-16}$. For $x = 10^{-10}$ the rounding error of $1 + x$ is up to $1.1 \times 10^{-16}$, a **relative** error of up to $10^{-6}$ in the part that matters. The logarithm faithfully reports $\ln$ of the wrong number: only about 7 of 16 digits survive. For $x < 1.1 \times 10^{-16}$, $1 + x$ rounds to exactly $1$ and $\ln(1 + x)$ returns $0$.

`math.log1p(x)` (and `np.log1p`) computes $\ln(1 + x)$ **without forming $1 + x$**, using the series for small $x$, so it is accurate to the last digit. Its partner `math.expm1(x)` computes $e^{x} - 1$ without the cancellation in the subtraction.

| Want | Naive | Stable |
|---|---|---|
| $\ln(1 + x)$, small $x$ | `math.log(1 + x)` | `math.log1p(x)` |
| $e^{x} - 1$, small $x$ | `math.exp(x) - 1` | `math.expm1(x)` |
| $(1 - p)^n$, tiny $p$ | `(1 - p) ** n` | `math.exp(n * math.log1p(-p))` |

### 2.4 Products become sums; sums need `logsumexp`

Products are easy in the log domain:

$$
\ln \prod_i p_i = \sum_i \ln p_i .
$$

**Sums** are harder. Given $a_i = \ln p_i$, we want $\ln \sum_i p_i = \ln \sum_i e^{a_i}$ — for example, to normalise probabilities or to add the likelihoods of alternative explanations. Exponentiating $a_i = -1000$ underflows to $0$, giving $\ln 0 = -\infty$; exponentiating $a_i = +1000$ overflows. The fix is to factor out the largest term $M = \max_i a_i$:

$$
\ln \sum_i e^{a_i} = M + \ln \sum_i e^{a_i - M} .
$$

Now every exponent $a_i - M$ is $\le 0$, so nothing overflows, and the largest term is exactly $e^0 = 1$, so the sum lies between $1$ and $n$ and its logarithm is safe. This is **logsumexp**. For two terms NumPy provides `np.logaddexp(a, b)` $= \ln(e^a + e^b)$, and `np.logaddexp.reduce(array)` folds it over a whole array.

## 3. Worked Examples

### Example 1 — when does a product of probabilities underflow?

$1000$ fair die rolls: $\log_{10} (1/6)^{1000} = -1000 \times 0.77815 = -778.15$. The smallest float has $\log_{10} \approx -323.3$, so the product underflows. In the log domain: $\ln P = 1000 \ln \tfrac16 = -1791.76$, no problem.

### Example 2 — the spacing of floats at $10^6$

$2^{19} = 524{,}288 \le 10^6 < 2^{20}$, so $e = 19$ and $\operatorname{ulp}(10^6) = 2^{19 - 52} = 2^{-33} \approx 1.16 \times 10^{-10}$. Relative spacing: $1.16 \times 10^{-16}$, between $2^{-53}$ and $2^{-52}$ as promised.

### Example 3 — $\ln(1 + 10^{-10})$

True value: $10^{-10} - 5 \times 10^{-21} + \cdots = 9.9999999995 \times 10^{-11}$. The naive `math.log(1 + 1e-10)` returns $1.00000008 \times 10^{-10}$: wrong in the 8th digit, because $1 + 10^{-10}$ was stored as $1.000000000100000008\ldots$. `math.log1p(1e-10)` returns $9.9999999995 \times 10^{-11}$.

### Example 4 — logsumexp by hand

$a = (-1000, -1001)$. Naively $e^{-1000} + e^{-1001} = 0 + 0$ in floats, so $\ln$ gives $-\infty$. With $M = -1000$:

$$
-1000 + \ln\left(e^{0} + e^{-1}\right) = -1000 + \ln(1.36788) = -1000 + 0.31326 = -999.68674 .
$$

### Example 5 — a tiny risk, repeated

A component fails with probability $p = 10^{-18}$ per use. Probability of no failure in $n = 10^{15}$ uses: $(1 - p)^n$. Naively $1 - 10^{-18}$ rounds to $1$, so the answer is $1$. Correctly, $(1-p)^n = e^{n \ln(1 - p)} \approx e^{-np} = e^{-0.001} = 0.9990005$ — computed with `math.exp(n * math.log1p(-p))`.

## 4. Python Examples

### 4.1 Underflow and the log-domain fix

In [ ]:
import math

import numpy as np

probabilities = np.full(1000, 1 / 6)

naive_product = 1.0
for p in probabilities:
    naive_product *= p

log_product = np.sum(np.log(probabilities))

print("naive product          :", naive_product)
print("sum of ln p            :", log_product)
print("as log10               :", log_product / math.log(10))
print("so P = 10 **", round(log_product / math.log(10), 2))

### 4.2 Inspecting the float format

`math.frexp` returns the mantissa in $[0.5, 1)$; `float.hex` shows the stored bits $(1.f)_2 \cdot 2^{e}$ in hexadecimal; `np.finfo` lists the landmarks.

In [ ]:
import math

import numpy as np

for x in [1.0, 0.1, 1e6, 6.02214076e23, 5e-324]:
    print(f"{x:<14g} hex: {x.hex():<24} frexp: {math.frexp(x)}")

info = np.finfo(np.float64)
print()
print("machine epsilon      :", info.eps, "=", 2.0**-52)
print("nextafter(1, 2) - 1  :", np.nextafter(1.0, 2.0) - 1.0)
print("largest float        :", info.max)
print("smallest normal      :", info.tiny)
print("smallest subnormal   :", np.nextafter(0.0, 1.0))
print("log10 of the range   :", round(math.log10(np.nextafter(0.0, 1.0)), 1), "to", round(math.log10(info.max), 1))

`0.1` is `0x1.999999999999ap-4`: $0.1$ has no finite binary expansion, so it is stored rounded — the reason `0.1 + 0.2 != 0.3`.

### 4.3 Spacing grows with the number; relative spacing does not

In [ ]:
import numpy as np

print("        x          spacing (ulp)     spacing / x")
for x in [1e-300, 1e-10, 1.0, 2.0, 1e6, 1e16, 1e300]:
    spacing = np.spacing(x)
    print(f"{x:>10.0e}    {spacing:>14.3e}    {spacing / x:.3e}")

print()
print("1e16 + 1 == 1e16 ?", 1e16 + 1 == 1e16, "  (the spacing at 1e16 is 2)")

### 4.4 `log1p` and `expm1`

In [ ]:
import math

print("     x        log(1+x) rel. error   log1p(x)          exp(x)-1 rel. error")
for x in [1e-3, 1e-6, 1e-10, 1e-15, 1e-17]:
    naive_log = math.log(1 + x)
    stable_log = math.log1p(x)
    naive_exp = math.exp(x) - 1
    stable_exp = math.expm1(x)
    log_error = abs(naive_log / stable_log - 1)
    exp_error = abs(naive_exp / stable_exp - 1)
    print(f"{x:>8.0e}    {log_error:>10.1e}          {stable_log:<16.10e}  {exp_error:>10.1e}")

p, n = 1e-18, 1e15
print()
print("(1 - p) ** n               :", (1 - p) ** n)
print("exp(n * log1p(-p))         :", math.exp(n * math.log1p(-p)))
print("probability of >= 1 failure:", -math.expm1(n * math.log1p(-p)), " (naive: 1 - (1-p)**n =", 1 - (1 - p) ** n, ")")

The naive relative error grows like $\varepsilon / x$: about $10^{-7}$ at $x = 10^{-10}$ and $11\%$ at $x = 10^{-15}$; at $x = 10^{-17}$ the naive formula returns $0$ (a $100\%$ error). The stable functions are exact to the last digit throughout.

### 4.5 `logsumexp`: naive, stable, and NumPy's

In [ ]:
import numpy as np


def logsumexp_naive(values):
    return np.log(np.sum(np.exp(values)))


def logsumexp(values):
    values = np.asarray(values, dtype=float)
    largest = np.max(values)
    return largest + np.log(np.sum(np.exp(values - largest)))


test_cases = {
    "small (-1000, -1001)": np.array([-1000.0, -1001.0]),
    "large (1000, 1001)  ": np.array([1000.0, 1001.0]),
    "ordinary (1, 2, 3)  ": np.array([1.0, 2.0, 3.0]),
}
with np.errstate(over="ignore", divide="ignore"):
    for label, values in test_cases.items():
        print(f"{label}: naive {logsumexp_naive(values):>12.6f}   stable {logsumexp(values):>12.6f}   np.logaddexp.reduce {np.logaddexp.reduce(values):>12.6f}")

### 4.6 Comparing two models in the log domain

Is a sequence of 2000 die rolls more likely under a fair die or under a die loaded towards six ($P(6) = 0.25$, others $0.15$)? The likelihoods are both about $10^{-1550}$ — zero in floats — but the **log-likelihood ratio** is an ordinary number. To turn the two log-likelihoods into posterior probabilities (equal priors), divide each likelihood by their sum: subtract `logaddexp`.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=5)
loaded_probabilities = np.array([0.15, 0.15, 0.15, 0.15, 0.15, 0.25])
fair_probabilities = np.full(6, 1 / 6)
rolls = rng.choice(6, size=2000, p=loaded_probabilities)

likelihood_fair = np.prod(fair_probabilities[rolls])
likelihood_loaded = np.prod(loaded_probabilities[rolls])
log_likelihood_fair = np.sum(np.log(fair_probabilities[rolls]))
log_likelihood_loaded = np.sum(np.log(loaded_probabilities[rolls]))

print("products            :", likelihood_fair, likelihood_loaded)
print("log-likelihoods     :", round(log_likelihood_fair, 2), round(log_likelihood_loaded, 2))
print("log-likelihood ratio:", round(log_likelihood_loaded - log_likelihood_fair, 2))

log_evidence = np.logaddexp(log_likelihood_fair, log_likelihood_loaded)
print("posterior P(loaded) :", np.exp(log_likelihood_loaded - log_evidence))
print("posterior P(fair)   :", np.exp(log_likelihood_fair - log_evidence))

## 5. Visualizations

### 5.1 A toy float system: crowded on a linear axis, even on a log axis

Floats with only 3 fraction bits ($8$ values per binade) and exponents $-3$ to $3$. On a linear axis they bunch up near zero; on a logarithmic axis they are almost evenly spaced, with the same pattern repeating in every binade.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fraction_bits = 3
toy_floats = np.array(sorted(
    (1 + j / 2**fraction_bits) * 2.0**exponent
    for exponent in range(-3, 4)
    for j in range(2**fraction_bits)
))

fig, (ax_linear, ax_log) = plt.subplots(2, 1, figsize=(11, 4))
ax_linear.plot(toy_floats, np.zeros_like(toy_floats), "|", markersize=20, color="tab:blue")
ax_linear.set_title("Toy floats (3 fraction bits) on a linear axis")
ax_linear.set_yticks([])
ax_linear.set_xlabel("x")

ax_log.plot(toy_floats, np.zeros_like(toy_floats), "|", markersize=20, color="tab:orange")
ax_log.set_xscale("log", base=2)
ax_log.set_title("The same floats on a logarithmic axis (base 2)")
ax_log.set_yticks([])
ax_log.set_xlabel("x (log scale)")
ax_log.grid(True, which="major", axis="x", alpha=0.5)
plt.tight_layout()
plt.show()

print("number of toy floats:", len(toy_floats), " smallest", toy_floats[0], " largest", toy_floats[-1])

### 5.2 The spacing of real doubles

Left: `np.spacing(x)` against $x$ on log–log axes — a staircase that doubles at every power of two and follows the line $x \cdot 2^{-52}$. Right: the relative spacing `np.spacing(x) / x`, a sawtooth between $2^{-53}$ and $2^{-52}$: constant relative precision, as on the slide rule.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x_wide = np.geomspace(1e-6, 1e6, 3000)
x_narrow = np.geomspace(0.25, 16, 3000)

fig, (ax_spacing, ax_relative) = plt.subplots(1, 2, figsize=(13, 4.5))
ax_spacing.loglog(x_wide, np.spacing(x_wide), label="np.spacing(x)")
ax_spacing.loglog(x_wide, x_wide * 2.0**-52, "--", label=r"$x \cdot 2^{-52}$")
ax_spacing.set_xlabel("x (log scale)")
ax_spacing.set_ylabel("gap to the next float (log scale)")
ax_spacing.set_title("Absolute spacing grows with x")
ax_spacing.grid(True, which="both", alpha=0.3)
ax_spacing.legend()

ax_relative.semilogx(x_narrow, np.spacing(x_narrow) / x_narrow, color="tab:orange", base=2)
ax_relative.axhline(2.0**-52, color="gray", linestyle=":", label=r"$2^{-52}$")
ax_relative.axhline(2.0**-53, color="gray", linestyle="--", label=r"$2^{-53}$")
ax_relative.set_xlabel("x (log scale, base 2)")
ax_relative.set_ylabel("spacing / x")
ax_relative.set_title("Relative spacing: a sawtooth, one tooth per binade")
ax_relative.grid(True, which="both", alpha=0.3)
ax_relative.legend()
plt.tight_layout()
plt.show()

### 5.3 Naive versus stable: $\ln(1+x)$ and $e^x - 1$

Relative error of the naive formulas against `log1p` / `expm1`, for $x$ from $10^{-18}$ to $1$ (both axes logarithmic). The error grows like $\varepsilon / (2x)$ (dashed line) as $x$ shrinks, and reaches $100\%$ once $1 + x$ rounds to $1$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.geomspace(1e-18, 1, 400)
log_error = np.abs(np.log(1 + x) / np.log1p(x) - 1)
exp_error = np.abs((np.exp(x) - 1) / np.expm1(x) - 1)
floor = 1e-17                                  # show exact agreement at the bottom of the plot

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(x, np.maximum(log_error, floor), ".", markersize=3, label="log(1 + x) vs log1p(x)")
ax.loglog(x, np.maximum(exp_error, floor), ".", markersize=3, label="exp(x) - 1 vs expm1(x)")
ax.loglog(x, np.minimum(2.0**-53 / x, 1), "k--", label=r"$\varepsilon / (2x)$")
ax.set_xlabel("x (log scale)")
ax.set_ylabel("relative error of the naive formula (log scale)")
ax.set_title("Cancellation: the smaller x, the fewer correct digits")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()

### 5.4 Where the naive `logsumexp` fails

$\ln(e^{a} + e^{a - 1})$ for a shift $a$ from $-1000$ to $1000$. The exact answer is $a + \ln(1 + e^{-1}) = a + 0.3133$. The naive formula is correct only while $e^{a}$ fits in a float — roughly $-745 < a < 709$ — and returns $-\infty$ or $+\infty$ outside; the stable version is correct everywhere. Just inside the left edge the naive points already scatter: $e^{a}$ is a subnormal there, with only a few significant bits left.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def logsumexp(values):
    largest = np.max(values)
    return largest + np.log(np.sum(np.exp(values - largest)))


shifts = np.linspace(-1000, 1000, 801)
with np.errstate(over="ignore", divide="ignore"):
    naive = np.array([np.log(np.exp(a) + np.exp(a - 1)) for a in shifts])
stable = np.array([logsumexp(np.array([a, a - 1])) for a in shifts])
exact_offset = np.log1p(np.exp(-1))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(shifts, stable - shifts, label="stable logsumexp - a", linewidth=3, alpha=0.6)
finite = np.isfinite(naive)
ax.plot(shifts[finite], naive[finite] - shifts[finite], ".", markersize=3, label="naive - a (where finite)")
ax.axhline(exact_offset, color="black", linestyle="--", linewidth=1, label=r"exact: $\ln(1 + e^{-1})$")
for edge in [shifts[finite][0], shifts[finite][-1]]:
    ax.axvline(edge, color="crimson", linestyle=":")
ax.text(-990, 0.2, "naive gives -inf", color="crimson")
ax.text(730, 0.2, "naive gives inf", color="crimson")
ax.set_ylim(0, 0.5)
ax.set_xlabel("shift a")
ax.set_ylabel(r"$\ln(e^a + e^{a-1}) - a$")
ax.set_title("logsumexp works for any shift; the naive formula only in the middle")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper center")
plt.show()

print("naive finite for a from", shifts[finite][0], "to", shifts[finite][-1])

## 6. Exercises

### Basic

1. Does $0.9^{10000}$ underflow? Decide with $\log_{10}$ first, then check in Python and give $\ln(0.9^{10000})$.
2. Compute `np.spacing(1e6)` by hand from the binade of $10^6$, then check.
3. Compare `math.log(1 + 1e-12)` with `math.log1p(1e-12)`. How many digits does the naive version get right?

### Intermediate

4. **Estimate it first, then compute.** After how many fair coin flips does the probability $0.5^n$ of a particular sequence underflow to exactly $0$? Estimate from the landmarks in section 2.2, then find it with a loop.
5. Compute `logsumexp([1000, 1000])` by hand and with your function and `np.logaddexp`. Why does the naive version fail?
6. The probability of exactly $5000$ heads in $10{,}000$ fair flips is $\binom{10000}{5000} 2^{-10000}$. Show that the direct formula fails in floats, and compute it in the log domain with `math.lgamma` ($\ln n! = $ `math.lgamma(n + 1)`). Compare with the approximation $1/\sqrt{\pi \cdot 5000}$.

### Advanced

7. Prove that the stable `logsumexp` cannot overflow and never takes $\ln 0$, for any finite input. Bound the sum $\sum e^{a_i - M}$.
8. Implement a stable softmax $\sigma(a)_i = e^{a_i} / \sum_j e^{a_j}$ as $\exp(a_i - \operatorname{logsumexp}(a))$. Test it on $a = (1000, 1001, 1002)$ and compare with the naive version.
9. The function $\ln(1 - e^{-x})$ for $x > 0$ (which appears in survival probabilities) loses accuracy for both tiny and large $x$ when computed naively. Implement the recommended version: `log(-expm1(-x))` for $x \le \ln 2$ and `log1p(-exp(-x))` for $x > \ln 2$. Compare with the naive formula at $x = 10^{-20}, 10^{-8}, 1, 40$.

## 7. Solutions

1. $\log_{10} 0.9^{10000} = 10000 \times (-0.045757) = -457.57 < -323.3$: it underflows to $0$. In the log domain $\ln(0.9^{10000}) = 10000 \ln 0.9 = -1053.61$.
2. $2^{19} \le 10^6 < 2^{20}$, so $\operatorname{ulp} = 2^{19-52} = 2^{-33} = 1.164 \times 10^{-10}$.
3. Naive: $1.0000889 \times 10^{-12}$; `log1p`: $9.999999999995 \times 10^{-13}$. Relative error $9 \times 10^{-5}$: only about 4 correct digits.
4. The smallest subnormal is $2^{-1074}$, so $0.5^{1074}$ is still representable. $0.5^{1075}$ lies exactly halfway between $0$ and $2^{-1074}$, and round-half-to-even sends it to $0$: the answer is $n = 1075$. (If you only count normal floats, $2^{-1022}$, the estimate is about $1022$.)
5. $M = 1000$, so the result is $1000 + \ln(e^0 + e^0) = 1000 + \ln 2 = 1000.693147$. The naive version computes $e^{1000} = \infty$, and $\ln \infty = \infty$.
6. $\binom{10000}{5000}$ is an exact Python integer with $3009$ digits, but multiplying it by a float raises `OverflowError`, and $0.5^{10000} = 0$. In logs: $\ln P = \ln 10000! - 2\ln 5000! - 10000 \ln 2$, giving $P = 0.0079786$. The approximation gives $0.0079788$.
7. Let $M = \max_i a_i$. Every exponent $a_i - M \le 0$, so each term is in $(0, 1]$ and none overflows; the term with $a_i = M$ is exactly $1$. So $1 \le \sum_i e^{a_i - M} \le n$, its logarithm lies in $[0, \ln n]$, and $\ln 0$ never occurs. Tiny terms may underflow to $0$, but they are negligible next to the $1$.
8. The stable softmax gives $(0.0900, 0.2447, 0.6652)$, the same as for $(0, 1, 2)$ — softmax only depends on differences. The naive version computes $\infty / \infty$ = `nan`. (The stable result sums to $1$ only within about $5 \times 10^{-14}$: subtracting $\operatorname{logsumexp}(a) \approx 1002.4$, whose ulp is $10^{-13}$, costs a few digits. Production code computes `exp(a - max(a))` and divides by its sum, which is slightly more accurate.)
9. At $x = 10^{-20}$: $1 - e^{-x}$ rounds to $0$ and the naive formula gives $-\infty$; the stable one gives $\ln(10^{-20}) = -46.05$. At $x = 40$: $e^{-40} \approx 4.25 \times 10^{-18}$ is lost in $1 - e^{-40} = 1$, so the naive answer is $0$; the stable one gives $-4.25 \times 10^{-18}$. At $x = 10^{-8}$ the naive $1 - e^{-x}$ keeps only about 8 correct digits, so the naive logarithm is wrong from the 10th digit on; at $x = 1$ both agree.

Numerical checks:

In [ ]:
import math

import numpy as np

print("Ex 1:", 0.9**10000, " log10:", round(10000 * math.log10(0.9), 2), " ln:", round(10000 * math.log(0.9), 2))
print("Ex 2:", np.spacing(1e6), 2.0**-33)
naive, stable = math.log(1 + 1e-12), math.log1p(1e-12)
print(f"Ex 3: naive {naive!r}, log1p {stable!r}, relative error {abs(naive / stable - 1):.1e}")

probability, flips = 1.0, 0
while probability > 0:
    probability *= 0.5
    flips += 1
print("Ex 4: underflows to 0 after", flips, "flips;  0.5**1074 =", 0.5**1074)


def logsumexp(values):
    values = np.asarray(values, dtype=float)
    largest = np.max(values)
    return largest + np.log(np.sum(np.exp(values - largest)))


with np.errstate(over="ignore"):
    print("Ex 5:", logsumexp([1000, 1000]), np.logaddexp(1000, 1000), " naive:", np.log(np.sum(np.exp(np.array([1000.0, 1000.0])))))

try:
    direct = math.comb(10000, 5000) * 0.5**10000
except OverflowError as error:
    direct = f"OverflowError ({error})"
log_probability = math.lgamma(10001) - 2 * math.lgamma(5001) - 10000 * math.log(2)
print("Ex 6: direct:", direct, "| 0.5**10000 =", 0.5**10000)
print(f"      log domain: {math.exp(log_probability):.7f}, approximation 1/sqrt(pi*5000) = {1 / math.sqrt(math.pi * 5000):.7f}, digits of comb: {len(str(math.comb(10000, 5000)))}")

values = np.array([1000.0, 1001.0, 1002.0])
stable_softmax = np.exp(values - logsumexp(values))
with np.errstate(over="ignore", invalid="ignore"):
    naive_softmax = np.exp(values) / np.sum(np.exp(values))
print("Ex 8: stable", np.round(stable_softmax, 4), " naive", naive_softmax, " sum", stable_softmax.sum())


def log1mexp(x):
    if x <= math.log(2):
        return math.log(-math.expm1(-x))
    return math.log1p(-math.exp(-x))


for x in [1e-20, 1e-8, 1.0, 40.0]:
    try:
        naive = math.log(1 - math.exp(-x))
    except ValueError:
        naive = float("-inf")
    print(f"Ex 9: x = {x:<6g} naive {naive!r:<24} stable {log1mexp(x)!r}")

## 8. Conclusion

- A double is $\pm(1.f)_2 \cdot 2^{e}$: scientific notation in base 2, with the exponent playing the role of $\lfloor \log_2 x \rfloor$. Floats are evenly spaced on a **logarithmic** axis: spacing $\operatorname{ulp}(x) \approx x \cdot 2^{-52}$, constant relative precision of about 16 digits, from $4.9 \times 10^{-324}$ to $1.8 \times 10^{308}$ ($\log_{10}$ from $-323.3$ to $308.3$).
- Products of many probabilities fall off the bottom of that range. Their logarithms do not: $\ln \prod p_i = \sum \ln p_i$ turns an underflowing product into an ordinary sum.
- Forming $1 + x$ for tiny $x$ throws digits away (catastrophic cancellation). Use `math.log1p` / `np.log1p` for $\ln(1 + x)$ and `math.expm1` / `np.expm1` for $e^x - 1$; write $(1 - p)^n$ as `exp(n * log1p(-p))`.
- Adding numbers that are stored as logs needs **logsumexp**: $\ln \sum e^{a_i} = M + \ln \sum e^{a_i - M}$ with $M = \max a_i$. NumPy's `np.logaddexp` and `np.logaddexp.reduce` do the same. It is the core of stable softmax, cross-entropy and Bayesian model comparison.
- The pattern of the whole module: Briggs, the slide rule, `math.log` and `logsumexp` all move a problem onto a logarithmic scale, where multiplication is addition and huge ranges become manageable.

**Next:** module 04 opens with [logarithmic differentiation](../04_logarithms_in_calculus/01_logarithmic_differentiation.ipynb): taking $\ln$ of both sides to differentiate $x^x$ and messy products.